In [1]:
import torch
import torch.nn as nn

In [2]:
torch.manual_seed(42)

In [3]:
X = torch.randn(400, 6)

# shape = [400]
true_logit = (
    1.2 * X[:, 0]
    - 1.8 * X[:, 1]
    + 0.5 * X[:, 2]
    + 0.9 * X[:, 3]
    - 0.4 * X[:, 4]
    + 0.2 * X[:, 5]
    + 0.4 * torch.randn(400)
)

# [400] 을 [400, 1] 형태로 변환하기
y = (true_logit > 0).float().unsqueeze(1)

print(true_logit.shape)

torch.Size([400])


In [4]:
from torch.utils.data import DataLoader, TensorDataset, random_split

dataset = TensorDataset(X, y)

train_dataset, val_dataset = random_split(
    dataset,
    [320, 80],
    generator=torch.Generator().manual_seed(42)
)

In [5]:
train_data_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_data_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [6]:
class BinaryClassifier(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = torch.nn.Linear(6, 10)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(10, 1)

    def forward(self, X):
        X = self.fc1(X)
        X = self.relu(X)
        X = self.fc2(X)

        return X

In [7]:
model = BinaryClassifier()

loss_fn = torch.nn.BCEWithLogitsLoss()

# 등록된 tensor들에 대해 미분해서 학습률만큼 grad에 빼주는 객체
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.05
)

In [8]:
loss_fn

BCEWithLogitsLoss()

In [10]:
model

BinaryClassifier(
  (fc1): Linear(in_features=6, out_features=10, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=10, out_features=1, bias=True)
)

In [16]:
list(model.parameters())[0][0][0]

tensor(0.1889, grad_fn=<SelectBackward0>)

In [11]:
optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.05
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [9]:
epochs = 50

for epoch in range(1, epochs+1):

    model.train()

    # 학습
    train_loss_sum = 0

    for X_batch, y_batch in train_data_loader:
        # 예측
        logits = model(X_batch)

        # 손실 점수
        loss = loss_fn(logits, y_batch)

        # 기존 grad 제거해주기
        optimizer.zero_grad()

        # 역전파를 통해 grad 새로 넣어주기
        # loss에 등록되어있는 logits에 대해 편미분을 실시하여 grad를 넣어주기
        loss.backward()

        # 등록된 grad를 기반으로 학습시켜주기
        optimizer.step()

        # 학습동안 존재한 loss 들 모두 합쳐주기
        train_loss_sum += loss.item() * X_batch.size(0)

    train_loss = train_loss_sum / len(train_dataset)

    model.eval()

    # 검증 (epoch 1 또는 10의 배수마다)
    if epoch == 1 or epoch % 10 == 0:
        with torch.inference_mode():
            val_loss_sum = 0
            correct = 0
            total = 0

            for X_batch, y_batch in val_data_loader:
                logits = model(X_batch)

                loss = loss_fn(logits, y_batch)

                val_loss_sum += loss.item() * y_batch.numel()

                probabilities = torch.sigmoid(logits)

                predictions = (probabilities >= 0.5).float()

                correct += (predictions == y_batch).sum().item()

                total += y_batch.numel()

            val_loss = val_loss_sum / total
            val_accuracy = correct / total

            print(
                f"epoch={epoch:3d} "
                f"train_loss={train_loss:.4f} "
                f"val_loss={val_loss:.4f} "
                f"val_acc={val_accuracy:.3f}"
            )

epoch=  1 train_loss=0.7046 val_loss=0.6997 val_acc=0.487
epoch= 10 train_loss=0.5175 val_loss=0.5306 val_acc=0.825
epoch= 20 train_loss=0.3040 val_loss=0.3272 val_acc=0.900
epoch= 30 train_loss=0.2160 val_loss=0.2392 val_acc=0.925
epoch= 40 train_loss=0.1827 val_loss=0.2080 val_acc=0.912
epoch= 50 train_loss=0.1667 val_loss=0.1931 val_acc=0.900
